In [12]:
using Revise
using TTNKit
using Test
using ITensors
using PyPlot
using LinearAlgebra
using LaTeXStrings
using KrylovKit
using Printf, BenchmarkTools, Random

In [13]:
create_wavefunction(sz::NTuple{D, Int}) where{D} = create_wavefunction(ComplexF64, sz)
create_wavefunction(elT, sz::NTuple{D,Int}) where{D} = normalize!(randn(elT, sz))

create_wavefunction (generic function with 2 methods)

In [33]:
function patron_application!(ttn::TreeTensorNetwork, wf_coefs::Array, op_ins::String; maxdim::Int = maxlinkdim(ttn), normalize::Bool = true)
    net = TTNKit.network(ttn)
    lat = TTNKit.physical_lattice(net)

    all(size(lat) .== size(wf_coefs)) || error("Trying to apply a patron wavefunction of dimensionality $(size(wf_coefs)) to a TTN defined on a lattice $(size(lat))")


    patron_mpo = wf_mpo(wf_coefs, net, op_ins)
    
    for p in eachindex(TTNKit.lattice(net, 1))
        ttn = TTNKit.move_ortho!(ttn, (1,p))
        pc = TTNKit.child_nodes(net, (1, p))
        Tpat = map(j -> patron_mpo[j], last.(pc))
        
        pr = TTNKit.parent_node(net, (1,p))
        #println("Site $p has parent ",pr)
        #println("Child nodes with lattice coords are $(pc[1][2]) at $(TTNKit.coordinate(lat,pc[1][2])) and $(pc[2][2]) at $(TTNKit.coordinate(lat,pc[2][2]))")
        Tt  = ttn[(1, p)]

        rind = TTNKit.commonind(Tt, ttn[pr])
        linds = TTNKit.uniqueinds(Tt, rind)
        Tn = TTNKit.noprime(TTNKit.contract(Tt, Tpat...))

        #println("Multiplication worked")
        #if p == 2
        #    return Tn,linds,rind
        #end
        A, R = TTNKit.factorize(Tn, linds, maxdim = maxdim, tags = TTNKit.tags(rind))
        
        #=
        println("Physical Site $p")
        println("Sizes: ",", Tn",length(TTNKit.inds(Tn)),", R",length(TTNKit.inds(R)),", Pr",length(TTNKit.inds(ttn[pr])),", A",length(TTNKit.inds(A)))
        println(", Tn",TTNKit.tags.(TTNKit.inds(Tn)),", R",TTNKit.tags.(TTNKit.inds(R)),", Pr",TTNKit.tags.(TTNKit.inds(ttn[pr])),", A",TTNKit.tags.(TTNKit.inds(A)))
        =#
        ttn[(1,p)] = A
        ttn[pr] = ttn[pr] * R    
        
        # moving the orhto center
        ttn.ortho_center[1] = pr[1]
        ttn.ortho_center[2] = pr[2]
        # this has to be deleted in the future... dont need this anymore
        ttn.ortho_direction[1][p] = TTNKit.number_of_child_nodes(net, (1,p)) + 1
        ttn.ortho_direction[pr[1]][pr[2]] = -1
    end


    # now move to the higher layers layer by layer, excluding the top node
    TTNKit.ITensors.set_warn_order(20)
    for ll in 2:TTNKit.number_of_layers(net)-1
        for p in 1:TTNKit.number_of_tensors(net, ll)
            pp = (ll, p)
            ttnc = TTNKit.move_ortho!(ttn, pp)
            
            Tt = ttn[pp]

            pr = TTNKit.parent_node(net, pp)

            pc = TTNKit.child_nodes(net, pp)
            
            linds = map(p -> TTNKit.commonind(Tt, ttn[p]), pc)
            ptags = "Link,nl=$(ll),np=$(p)"#tags(commonind(Tt, ttn[pr]))
        
            A, R = factorize(Tt, linds; maxdim = maxdim, tags = ptags)
            #=
            println(ll,", ",p)
            println("With Parent $pr")
            println("Sizes: ",", R",length(TTNKit.inds(R)),", Tt",length(TTNKit.inds(Tt)),", A",length(TTNKit.inds(A)))
            println(", R",TTNKit.tags.(TTNKit.inds(R)),", Tt",TTNKit.tags.(TTNKit.inds(Tt)),", A",TTNKit.tags.(TTNKit.inds(A)))
            =#
            ttn[pp] = A
            ttn[pr] = ttn[pr] * R
        
            # moving the orhto center
            ttn.ortho_center[1] = pr[1]
            ttn.ortho_center[2] = pr[2]
            # this has to be deleted in the future... dont need this anymore
            ttn.ortho_direction[ll][p] = TTNKit.number_of_child_nodes(net, (ll,p)) + 1
            ttn.ortho_direction[pr[1]][pr[2]] = -1
        end
    end

    if normalize
        tpnd = (TTNKit.number_of_layers(net), 1) 
        T = ttn[tpnd]
        ttn[tpnd] = T/norm(T)
    end

    return ttn
end

patron_application! (generic function with 1 method)

In [15]:
function get_2dttn_path(size) # written by ChapGPT 29.05.2023
    #lattice = [(i, j) for i in 1:size, j in 1:size]
    path = []

    function traverse_lattice(x_start, y_start, x_end, y_end)
        if x_start == x_end && y_start == y_end
            push!(path, (y_start, x_start))
        else
            x_mid = (x_start + x_end) ÷ 2
            y_mid = (y_start + y_end) ÷ 2
            traverse_lattice(x_start, y_start, x_mid, y_mid)
            traverse_lattice(x_mid + 1, y_start, x_end, y_mid)
            traverse_lattice(x_start, y_mid + 1, x_mid, y_end)
            traverse_lattice(x_mid + 1, y_mid + 1, x_end, y_end)
        end
    end

    traverse_lattice(1, 1, size, size)
    return path
end

function get_site_number(x, y, side_length)
    site_number = (y - 1) * side_length + x
    if site_number > side_length^2
    	println("ERROR: Outside Square")
    	return Int(site_number)
    end
    return Int(site_number)
end

function get_2d_mapping(size)
    path = get_2dttn_path(size)
    map2d::Vector{Int64} = []
    for i in 1:length(path)
        site_idx = Int(get_site_number(path[i][2],path[i][1],size))
        append!(map2d,[site_idx])
    end
    return map2d
end

function construct_first_layer(mpo_wrap,net)
    n_sites = TTNKit.number_of_sites(net)
    n_tensors = TTNKit.number_of_tensors(net) + n_sites
    bEnvironment = Vector{Vector{Vector{ITensor}}}(undef, 1) 
    bIndices = Vector{Vector{Vector{Vector{Int}}}}(undef, 1) 
    
    ham = mpo_wrap.data
    # First layer

    bEnvironment = map(eachindex(net,1)) do pp
        chdnds = TTNKit.child_nodes(net, (1,pp))
        map(1:TTNKit.number_of_child_nodes(net, (1,pp))) do nn
          ham[TTNKit.inverse_mapping(mpo_wrap.mapping)[chdnds[nn][2]]]
        end
    end
    # first layer
    virt_leg = mpo_wrap.mapping
    bIndices = map(eachindex(net,1)) do pp
        chdnds = TTNKit.child_nodes(net, (1,pp))
        map(1:TTNKit.number_of_child_nodes(net, (1,pp))) do nn
            chdid_virt = virt_leg[chdnds[nn][2]]
            int_leg = TTNKit.internal_index_of_legs(net, (1, pp))

            if isone(chdid_virt)
                return [int_leg[nn] + n_tensors, int_leg[nn], -chdid_virt]
            elseif chdid_virt == TTNKit.number_of_sites(net)
                return [-chdid_virt+1, int_leg[nn] + n_tensors, int_leg[nn]]
            else
                return [-chdid_virt+1, int_leg[nn] + n_tensors, int_leg[nn], -chdid_virt]
            end 
        end
    end
    
    return bIndices,bEnvironment
end


construct_first_layer (generic function with 1 method)

In [28]:
function wf_mpo(wf, net, op_ins)
    ampo = TTNKit.OpSum()
    lat = TTNKit.physical_lattice(net)
    for pci in keys(wf)
        pt = to_indices(wf, (pci,))

        plin = TTNKit.linear_ind(lat, pt)
        ampo += (wf[pci], op_ins, plin)
    end
    #reg_mpo = TTNKit.MPO(ampo,TTNKit.siteinds(lat))
    mpo_wrap = TTNKit.Hamiltonian(ampo,lat;mapping=TTNKit.hilbert_curve(lat))#get_2d_mapping(size(lat)[1]))
    rez_indices, rez_data = construct_first_layer(mpo_wrap,net)
    remapped_mpo::Vector{ITensor} = []
    for i in 1:size(rez_data,1)
        for j in 1:size(rez_data[i],1)
            append!(remapped_mpo,[rez_data[i][j]])
        end
    end
    return remapped_mpo
end

wf_mpo (generic function with 1 method)

In [32]:
#=
begin n_layer = 10; nd = TTNKit.ITensorNode; conserve_qns = true; maxdim = 4; normalize = true

    #t = 1; phi = 0.15*2*π; Nbosons = 12; vpinning = 2.5
    net = TTNKit.BinaryRectangularNetwork(n_layer, nd, "Boson"; conserve_qns = conserve_qns)
    #net = TTNKit.BinaryChainNetwork(n_layer, nd, "Boson"; conserve_qns = conserve_qns)
    lat = TTNKit.physical_lattice(net)
    println("Built Network")
    nsites = TTNKit.number_of_sites(net)
    
    
    states = fill("0", nsites) 
    
    ttn = TTNKit.ProductTreeTensorNetwork(net, states)
    ttn0 = deepcopy(ttn)
    
    wf_coefs = create_wavefunction(Float64, size(lat))
    #wf_coefs = randn(Float64, size(lat))
    #println("Made TTN and Wavefunc")
    ttn = patron_application!(ttn, wf_coefs, "Adag"; maxdim = maxdim, normalize = normalize)

    n_ex = real.(TTNKit.expect(ttn, "n"))
end
imshow(n_ex)
colorbar()
sum(n_ex)=#

#let   wf_coefs.^2
#end

In [7]:
#eachind = [p for p in eachindex(TTNKit.lattice(net,1))]

In [8]:
#=pc = []
for i in 1:length(eachind)
    p = eachind[i]
    ttn = TTNKit.move_ortho!(ttn, (1,p))
    pc_val = last.(TTNKit.child_nodes(net,(1,p)))
    append!(pc,[pc_val])
end
pc=#

In [9]:
#patron_final = wf_mpo(wf_coefs, net, "Adag");

In [10]:
#=flatver1::Vector{ITensor} = []
for i in 1:size(patron_final[2],1)
    for j in 1:size(patron_final[2][i],1)
        append!(flatver1,[patron_final[2][i][j]])
    end
end
flatver1=#